In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("../data/row/subset.csv")
df.head(20)

,id,text,Not Toxic,Hard to Say,Toxic,Very Toxic,category_卑語,category_差別,category_迷惑行為,category_猥褻,category_出会い・プライバシー侵害,category_違法行為,category_偏向表現,annotation_num
0,0,いわゆる天使だろ,2,0,0,1,0,0,0,0,0,0,0,3
1,3,底知れないカリスマ性があるよな,4,0,0,0,0,0,0,0,0,0,0,4
2,9,ありがとう,5,0,0,0,0,0,0,0,0,0,0,5
3,10,地味にこれが一番きつい,3,1,0,0,0,0,0,0,0,0,0,4
4,14,ヤクザより怖い,1,3,0,0,1,3,2,0,0,1,0,4
5,15,それこそ悪魔だわ,3,1,0,0,0,1,0,0,0,0,1,4
6,16,ずるいわ,4,0,0,0,0,0,0,0,0,0,0,4
7,17,こういう明らかに流行るゲームはせめてスタートダッシュかまさないとな,4,0,0,0,0,0,0,0,0,0,0,4
8,27,あっためかえしかよ,3,1,0,0,0,0,0,0,0,0,0,4
9,28,ナイス壁,4,0,0,0,0,0,0,0,0,0,0,4


In [3]:
df.dtypes

id                       int64
text                       str
Not Toxic                int64
Hard to Say              int64
Toxic                    int64
Very Toxic               int64
category_卑語              int64
category_差別              int64
category_迷惑行為            int64
category_猥褻              int64
category_出会い・プライバシー侵害    int64
category_違法行為            int64
category_偏向表現            int64
annotation_num           int64
dtype: object

In [4]:
df.columns.tolist()

['id',
 'text',
 'Not Toxic',
 'Hard to Say',
 'Toxic',
 'Very Toxic',
 'category_卑語',
 'category_差別',
 'category_迷惑行為',
 'category_猥褻',
 'category_出会い・プライバシー侵害',
 'category_違法行為',
 'category_偏向表現',
 'annotation_num']

In [5]:
# 有害レベル
level_columns = [
    "Not Toxic",
    "Hard to Say",
    "Toxic",
    "Very Toxic"
]

df["level"] = df[level_columns].idxmax(axis=1)


# 適切 / 不適切
classification_map = {
    "Not Toxic": 0,
    "Toxic": 1,
    "Very Toxic": 1
}

df["classification"] = df["level"].map(classification_map)



In [6]:
# 有害カテゴリ
category_columns = [
    "category_卑語",
    "category_差別",
    "category_迷惑行為",
    "category_猥褻",
    "category_出会い・プライバシー侵害",
    "category_違法行為",
    "category_偏向表現"
]

print(df[category_columns].max(axis=1).value_counts())

df["category"] = df[category_columns].idxmax(axis=1)

# 適切な文章にはカテゴリを付けない
df.loc[
    df["classification"] == 0,
    "category"
] = None


# 必要な列だけ残す
df = df[
    [
        "text",
        "level",
        "classification",
        "category"
    ]
]


# 保存
df.to_csv(
    "../data/processed/toxic_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)


print("整形後データ:")
print(df.head())

print("保存しました")

0    319
1     68
2     32
3     11
4      5
5      2
Name: count, dtype: int64
整形後データ:
              text        level  classification     category
0         いわゆる天使だろ    Not Toxic             0.0          NaN
1  底知れないカリスマ性があるよな    Not Toxic             0.0          NaN
2            ありがとう    Not Toxic             0.0          NaN
3      地味にこれが一番きつい    Not Toxic             0.0          NaN
4          ヤクザより怖い  Hard to Say             NaN  category_差別
保存しました


In [7]:
df.head(20)

,text,level,classification,category
0,いわゆる天使だろ,Not Toxic,0.0,NaN
1,底知れないカリスマ性があるよな,Not Toxic,0.0,NaN
2,ありがとう,Not Toxic,0.0,NaN
3,地味にこれが一番きつい,Not Toxic,0.0,NaN
4,ヤクザより怖い,Hard to Say,NaN,category_差別
5,それこそ悪魔だわ,Not Toxic,0.0,NaN
6,ずるいわ,Not Toxic,0.0,NaN
7,こういう明らかに流行るゲームはせめてスタートダッシュかまさないとな,Not Toxic,0.0,NaN
8,あっためかえしかよ,Not Toxic,0.0,NaN
9,ナイス壁,Not Toxic,0.0,NaN


In [8]:
# 欠損値の数
(df["level"] == "Hard to Say").sum()

np.int64(36)

# Only Classを読み込んで学習

In [9]:
import pandas as pd

df = pd.read_csv("../data/processed/toxic_dataset_only_class.csv")
df.head()

,text,level,classification
0,いわゆる天使だろ,Not Toxic,0
1,底知れないカリスマ性があるよな,Not Toxic,0
2,ありがとう,Not Toxic,0
3,地味にこれが一番きつい,Not Toxic,0
4,ヤクザより怖い,Hard to Say,0


In [10]:
df.dtypes

text                str
level               str
classification    int64
dtype: object

In [11]:
df["classification"].value_counts()

classification
0    377
1     60
Name: count, dtype: int64

In [12]:
df.isna().sum()

text              0
level             0
classification    0
dtype: int64

In [13]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["classification"]
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["classification"]
)

print("train:", len(train_df))
print("valid:", len(valid_df))
print("test :", len(test_df))

train: 349
valid: 44
test : 44


In [14]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df[["text", "classification"]]
)

valid_dataset = Dataset.from_pandas(
    valid_df[["text", "classification"]]
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "classification"]]
)

print("="*80)
print(train_dataset)
print(train_dataset[0])
print("="*80)
print(valid_dataset)
print(valid_dataset[0])
print("="*80)
print(test_dataset)
print(test_dataset[0])

/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['text', 'classification', '__index_level_0__'],
    num_rows: 349
})
{'text': '安定性もあるしな', 'classification': 0, '__index_level_0__': 371}
Dataset({
    features: ['text', 'classification', '__index_level_0__'],
    num_rows: 44
})
{'text': 'そのクラスになると増やすためじゃなくて減らさないためにやってそうだな', 'classification': 0, '__index_level_0__': 83}
Dataset({
    features: ['text', 'classification', '__index_level_0__'],
    num_rows: 44
})
{'text': '去年は普通に戦力でしたし', 'classification': 0, '__index_level_0__': 105}


In [15]:
from transformers import AutoTokenizer

MODEL_NAME = "cl-tohoku/bert-base-japanese-v3"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [16]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [17]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_valid = valid_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

print(tokenized_train[0])
print(tokenized_valid[0])
print(tokenized_test[0])

Parameter 'function'=<function tokenize_function at 0xf8c006eef380> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only shown once. Subsequent hashing failures won't be shown.
Map: 100%|█████████| 44/44 [00:00<00:00, 5818.81 examples/s]

{'text': '安定性もあるしな', 'classification': 0, '__index_level_0__': 371, 'input_ids': [2, 14207, 2183, 484, 12485, 441, 460, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}
{'text': 'そのクラスになると増やすためじゃなくて減らさないためにやってそうだな', 'classification': 0, '__index_level_0__': 83, 'input_ids': [2, 12

In [18]:
text = train_dataset[0]["text"]

print("元の文章:")
print(text)

print("\nトークン:")
print(tokenizer.tokenize(text))

print("\nID:")
print(tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text)))

print("\nTokenizerの出力:")
print(tokenizer(
    text,
    padding="max_length",
    truncation=True,
    max_length=128
))

元の文章:
安定性もあるしな

トークン:
['安定', '性', 'も', 'ある', 'し', 'な']

ID:
[14207, 2183, 484, 12485, 441, 460]

Tokenizerの出力:
{'input_ids': [2, 14207, 2183, 484, 12485, 441, 460, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


## BERTでの学習

In [19]:
from transformers import AutoModelForSequenceClassification

In [20]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

Loading weights: 100%|█| 199/199 [00:00<00:00, 33636.92it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cl-tohoku/bert-base-japanese-v3
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archite

In [21]:
def rename_label(dataset):
    return dataset.rename_column(
        "classification",
        "labels"
    )

tokenized_train = rename_label(tokenized_train)
tokenized_valid = rename_label(tokenized_valid)
tokenized_test = rename_label(tokenized_test)

In [22]:
print(tokenized_train[0])

{'text': '安定性もあるしな', 'labels': 0, '__index_level_0__': 371, 'input_ids': [2, 14207, 2183, 484, 12485, 441, 460, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [23]:
from transformers import TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import Trainer

In [25]:
training_args = TrainingArguments(
    output_dir="./models/toxic-bert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    # logging_dir="./logs",
    logging_steps=50,
    load_best_model_at_end=True
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [26]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    compute_metrics=compute_metrics
)

trainer.train()

/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.396925,0.863636,0.000000,0.000000,0.000000
2,0.417153,0.403445,0.863636,0.000000,0.000000,0.000000
3,0.347124,0.364432,0.863636,0.000000,0.000000,0.000000
4,0.290174,0.443171,0.840909,0.000000,0.000000,0.000000
5,0.118344,0.517013,0.840909,0.000000,0.000000,0.000000


/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
Writing model shards: 100%|███| 1/1 [00:01<00:00,  1.01s/it]
/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
Writing model shards: 100%|███| 1/1 [00:02<00:00,  2.18s/it]
/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then devic

TrainOutput(global_step=220, training_loss=0.2769021673636003, metrics={'train_runtime': 19011.0583, 'train_samples_per_second': 0.092, 'train_steps_per_second': 0.012, 'total_flos': 114782197900800.0, 'train_loss': 0.2769021673636003, 'epoch': 5.0})

In [27]:
test_result = trainer.evaluate(
    tokenized_test
)

print(test_result)

/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.118344,0.338430,5,0.840909,0.000000,0.000000,0.000000


{'eval_loss': 0.3384300172328949, 'eval_accuracy': 0.8409090909090909, 'eval_precision': 0.0, 'eval_recall': 0.0, 'eval_f1': 0.0}


In [28]:
print("=== train ===")
print(train_df["classification"].value_counts())

print("\n=== validation ===")
print(valid_df["classification"].value_counts())

print("\n=== test ===")
print(test_df["classification"].value_counts())

=== train ===
classification
0    301
1     48
Name: count, dtype: int64

=== validation ===
classification
0    38
1     6
Name: count, dtype: int64

=== test ===
classification
0    38
1     6
Name: count, dtype: int64


In [29]:
from sklearn.metrics import confusion_matrix

predictions = trainer.predict(tokenized_valid)

pred_labels = np.argmax(
    predictions.predictions,
    axis=-1
)

true_labels = predictions.label_ids

print(confusion_matrix(true_labels, pred_labels))

/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


/home/kobat/bbs-with-AI-monitoring/.env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


[[38  0]
 [ 6  0]]
